# 05 - Silver: TRM (Tasa de Cambio Representativa del Mercado)

Trabajo Práctico 2 - Transformación Bronze → Silver con reglas de calidad (DQL)

## Objetivo

Tipar y validar `Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api` (serie diaria de la TRM). Las filas que pasan todas las reglas de calidad van a `Datos_Empresas.silver.tasa_cambio`; las que incumplen al menos una van a `Datos_Empresas.dql.rechazados_trm`.

## Por qué se agrega TRM como cuarta fuente

El TP1 dejó TRM fuera de las "tablas principales" porque no tiene una llave de unión a nivel de fila con RUES/SECOP/CIIU. Pero para responder la pregunta de negocio *"¿influye la TRM en el cierre de empresas (cancelaciones de matrícula en RUES)?"* se necesita cruzar, en la Capa Gold, el valor **mensual promedio de la TRM** contra el **conteo mensual de cancelaciones** de RUES — una unión por periodo de tiempo, no por llave de negocio. Por eso TRM entra aquí como una cuarta fuente Silver liviana.

## Transformaciones de tipado

* `valor`: texto → `DOUBLE`.
* `vigenciadesde`, `vigenciahasta`: texto ISO (`yyyy-MM-ddTHH:mm:ss.SSS`) → `DATE`.
* `unidad`: se recortan espacios (`trim`).

## Reglas de calidad (rechazo)

| # | Regla | Justificación |
|---|---|---|
| 1 | `valor` nulo, no numérico o `<= 0` | Una tasa de cambio no puede ser cero ni negativa — regla de negocio básica. |
| 2 | `vigenciadesde` posterior a `vigenciahasta` | Rango de vigencia inconsistente: la fecha de inicio no puede ser posterior a la de fin. |

In [ ]:
%python
from pyspark.sql import functions as F

TABLA_BRONZE = "Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api"
TABLA_SILVER = "Datos_Empresas.silver.tasa_cambio"
TABLA_DQL = "Datos_Empresas.dql.rechazados_trm"

df_bronze = spark.table(TABLA_BRONZE)
print(f"Bronze: {df_bronze.count():,} filas")

---

## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Resultado real contra Bronze: Regla 1 (valor inválido) = 0, Regla 2 (rango de vigencia inválido) = 0. La TRM es una serie oficial del Banco de la República, llega limpia; las reglas quedan como salvaguarda para cargas futuras.

In [ ]:
%sql
-- Evidencia Regla 1 y 2
SELECT 
  SUM(CASE WHEN try_cast(valor AS DOUBLE) IS NULL OR try_cast(valor AS DOUBLE) <= 0 THEN 1 ELSE 0 END) AS regla1_valor_invalido,
  SUM(CASE WHEN try_to_timestamp(vigenciadesde, "yyyy-MM-dd'T'HH:mm:ss.SSS") > try_to_timestamp(vigenciahasta, "yyyy-MM-dd'T'HH:mm:ss.SSS") THEN 1 ELSE 0 END) AS regla2_rango_invalido
FROM Datos_Empresas.bronze.DE_Semiestructurado_TasaCambio_Api;

---

## Paso 1: Tipado y reglas de calidad

In [ ]:
%python
FORMATO_FECHA_ISO = "yyyy-MM-dd'T'HH:mm:ss.SSS"

# try_to_timestamp (no to_timestamp): con ANSI SQL activado, to_timestamp lanza
# excepción ante una fecha mal formada en vez de devolver NULL.
df_reglas = (
    df_bronze
    .withColumn("unidad", F.trim(F.col("unidad")))
    .withColumn("valor", F.col("valor").cast("double"))
    .withColumn("vigenciadesde", F.expr(f"CAST(try_to_timestamp(vigenciadesde, '{FORMATO_FECHA_ISO}') AS DATE)"))
    .withColumn("vigenciahasta", F.expr(f"CAST(try_to_timestamp(vigenciahasta, '{FORMATO_FECHA_ISO}') AS DATE)"))
    # Regla 1: valor inválido
    .withColumn(
        "_r1",
        F.when((F.col("valor").isNull()) | (F.col("valor") <= 0), F.lit("valor_invalido")),
    )
    # Regla 2: rango de vigencia inconsistente
    .withColumn(
        "_r2",
        F.when(F.col("vigenciadesde") > F.col("vigenciahasta"), F.lit("rango_vigencia_invalido")),
    )
    .withColumn("_motivo_rechazo", F.concat_ws("; ", F.col("_r1"), F.col("_r2")))
    .withColumn(
        "_motivo_rechazo",
        F.when(F.col("_motivo_rechazo") == "", F.lit(None)).otherwise(F.col("_motivo_rechazo")),
    )
)

---

## Paso 2: Separar aceptados (Silver) de rechazados (DQL) y persistir

In [ ]:
%python
columnas_auxiliares = ["_r1", "_r2"]

df_silver = (
    df_reglas.filter(F.col("_motivo_rechazo").isNull())
    .drop(*columnas_auxiliares, "_motivo_rechazo", "_ingested_at", "_source")
    .withColumn("_processed_at", F.current_timestamp())
)

df_dql = (
    df_reglas.filter(F.col("_motivo_rechazo").isNotNull())
    .drop(*columnas_auxiliares)
    .withColumn("_fecha_rechazo", F.current_timestamp())
)

total = df_bronze.count()
aceptados = df_silver.count()
rechazados = df_dql.count()
print(f"Bronze: {total:,} | Silver (aceptados): {aceptados:,} | DQL (rechazados): {rechazados:,}")
assert total == aceptados + rechazados, "Se perdieron o duplicaron filas en la separación Silver/DQL"

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_SILVER)
df_dql.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_DQL)

print(f"Escrito: {TABLA_SILVER}")
print(f"Escrito: {TABLA_DQL}")

---

## Paso 3: Validación

In [ ]:
%sql
SELECT COUNT(*) AS total_silver, MIN(vigenciadesde) AS desde, MAX(vigenciadesde) AS hasta
FROM Datos_Empresas.silver.tasa_cambio;

SELECT COUNT(*) AS total_dql FROM Datos_Empresas.dql.rechazados_trm;

---

## Siguiente paso

Continuar con `06_Gold_Empresas_Contratos.ipynb`.